In [ ]:
from ollama import chat
import sqlite3

MODEL = "qwen3:4b"
MAX_ITERATIONS = 5


def get_sales_summary() -> dict:
    """Return monthly sales and percentage change from SQLite."""

    with sqlite3.connect("research.db") as conn:
        rows = conn.execute("""
            SELECT month, units_sold
            FROM monthly_sales
            ORDER BY month DESC
            LIMIT 2
        """).fetchall()

    if len(rows) < 2:
        return {
            "error": "At least two months of sales data are required."
        }

    current_month, current_sales = rows[0]
    previous_month, previous_sales = rows[1]

    change_percent = (
        (current_sales - previous_sales) / previous_sales * 100
        if previous_sales != 0
        else None
    )

    return {
        "previous_month": previous_month,
        "previous_month_sales": previous_sales,
        "current_month": current_month,
        "current_month_sales": current_sales,
        "change_percent": round(change_percent, 2)
        if change_percent is not None
        else None,
    }


def get_inventory_status() -> dict:
    """Return current inventory and minimum stock levels."""

    with sqlite3.connect("research.db") as conn:
        row = conn.execute("""
            SELECT product_id, available_units, minimum_required_units
            FROM inventory
            WHERE product_id = ?
        """, ("PROD-001",)).fetchone()

    if row is None:
        return {"error": "Product PROD-001 was not found."}

    product_id, available, minimum = row

    return {
        "product_id": product_id,
        "available_units": available,
        "minimum_required_units": minimum,
        "stock_status": (
            "above_minimum" if available >= minimum else "below_minimum"
        ),
    }

tools = [
    get_sales_summary,
    get_inventory_status,
]

available_tools = {
    "get_sales_summary": get_sales_summary,
    "get_inventory_status": get_inventory_status,
}


def execute_tool(name: str) -> dict:
    """Execute a registered tool safely."""
    tool = available_tools.get(name)

    if tool is None:
        return {"error": f"Unknown tool: {name}"}

    try:
        return tool()
    except Exception as exc:
        return {"error": f"Tool execution failed: {exc}"}


def run_agent(question: str) -> str:
    messages = [
        {
            "role": "user",
            "content": question,
        }
    ]

    for iteration in range(MAX_ITERATIONS):
        response = chat(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        messages.append(response.message)

        tool_calls = response.message.tool_calls or []

        # No tool calls: the model has produced its final answer.
        if not tool_calls:
            return response.message.content or ""

        for call in tool_calls:
            name = call.function.name
            result = execute_tool(name)

            print(f"\nHerramienta ejecutada: {name}")
            print(f"Resultado: {result}")

            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": str(result),
            })

    return (
        "No se obtuvo una respuesta final dentro del límite "
        f"de {MAX_ITERATIONS} iteraciones."
    )


if __name__ == "__main__":
    question = (
        "Investiga la situación del producto. Consulta las ventas "
        "y el inventario, compara los resultados y distingue "
        "los hallazgos de las hipótesis. No inventes causas."
    )

    print("\nRespuesta final:")
    print(run_agent(question))